In [6]:
from pathlib import Path
import json
import numpy as np

from sentence_transformers import SentenceTransformer

from embedding_config import (
    MODEL_NAME,
    BATCH_SIZE,
    NORMALIZE_EMBEDDINGS,
    EMBEDDING_DIMENSION
)


# ============================================================
# Paths
# ============================================================

CHUNKS_FILE = Path(
    "processed/chunks/chunks.json"
)

OUTPUT_DIR = Path(
    "processed/embeddings"
)

EMBEDDINGS_FILE = OUTPUT_DIR / "embeddings.npy"

METADATA_FILE = OUTPUT_DIR / "embedding_metadata.json"


# ============================================================
# Load chunks
# ============================================================

def load_chunks():

    with open(
        CHUNKS_FILE,
        "r",
        encoding="utf-8"
    ) as f:

        chunks = json.load(f)

    return chunks


# ============================================================
# Load model
# ============================================================

def load_embedding_model():

    print("Loading embedding model...")

    model = SentenceTransformer(
        MODEL_NAME
    )

    print(f"Model loaded: {MODEL_NAME}")

    return model


# ============================================================
# Generate embeddings
# ============================================================

def generate_embeddings(model, chunks):

    texts = [
        chunk["text"]
        for chunk in chunks
    ]

    print("\nGenerating embeddings...")
    print(f"Number of chunks: {len(texts)}")
    print(f"Batch size: {BATCH_SIZE}")

    embeddings = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        normalize_embeddings=NORMALIZE_EMBEDDINGS,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    return embeddings


# ============================================================
# Validate embeddings
# ============================================================

def validate_embeddings(embeddings, chunks):

    print("\n================================")
    print("EMBEDDING VALIDATION")
    print("================================")

    print(
        f"Number of embeddings: "
        f"{len(embeddings)}"
    )

    print(
        f"Embedding shape: "
        f"{embeddings.shape}"
    )

    print(
        f"Data type: "
        f"{embeddings.dtype}"
    )

    expected_shape = (
        len(chunks),
        EMBEDDING_DIMENSION
    )

    if embeddings.shape != expected_shape:

        raise ValueError(
            f"Unexpected embedding shape. "
            f"Expected {expected_shape}, "
            f"got {embeddings.shape}"
        )

    print("Shape validation: PASSED")


# ============================================================
# Save embeddings
# ============================================================

def save_embeddings(embeddings):

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    np.save(
        EMBEDDINGS_FILE,
        embeddings
    )

    print(
        f"\nEmbeddings saved to:\n"
        f"{EMBEDDINGS_FILE.resolve()}"
    )


# ============================================================
# Save metadata mapping
# ============================================================

def save_metadata(chunks):

    metadata = []

    for chunk in chunks:

        metadata.append({
            "chunk_id": chunk["chunk_id"],
            "metadata": chunk["metadata"],
            "text": chunk["text"]
        })

    with open(
        METADATA_FILE,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            metadata,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(
        f"Metadata saved to:\n"
        f"{METADATA_FILE.resolve()}"
    )


# ============================================================
# Main
# ============================================================

if __name__ == "__main__":

    print("================================")
    print("FLAME IN FREEFALL EMBEDDING")
    print("================================")

    # Load chunks
    chunks = load_chunks()

    print(
        f"\nLoaded {len(chunks)} chunks."
    )

    # Load model
    model = load_embedding_model()

    # Generate embeddings
    embeddings = generate_embeddings(
        model,
        chunks
    )

    # Validate
    validate_embeddings(
        embeddings,
        chunks
    )

    # Save vectors
    save_embeddings(
        embeddings
    )

    # Save mapping
    save_metadata(
        chunks
    )

    print("\n================================")
    print("EMBEDDING COMPLETE")
    print("================================")

FLAME IN FREEFALL EMBEDDING

Loaded 1200 chunks.
Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded: BAAI/bge-small-en-v1.5

Generating embeddings...
Number of chunks: 1200
Batch size: 32


Batches:   0%|          | 0/38 [00:00<?, ?it/s]


EMBEDDING VALIDATION
Number of embeddings: 1200
Embedding shape: (1200, 384)
Data type: float32
Shape validation: PASSED

Embeddings saved to:
C:\Users\User\Nasa Rag SYS\rag\embeddings\processed\embeddings\embeddings.npy
Metadata saved to:
C:\Users\User\Nasa Rag SYS\rag\embeddings\processed\embeddings\embedding_metadata.json

EMBEDDING COMPLETE
